In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
import sys
import os

PROJECT_ROOT = "/content/drive/MyDrive/isic-flagship-project"
os.chdir(PROJECT_ROOT)

In [ ]:


if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

!pip install langchain langchain-community langchain-openai faiss-cpu sentence-transformers -q

print(f"Working directory: {os.getcwd()}")


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/rag/rag_engine.py
"""
RAG engine for answering questions about the project
"""

import glob
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

class RAGEngine:
    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        self.vectorstore = None

    def index_project(self):
        documents = []


        for filepath in glob.glob("src/**/*.py", recursive=True):
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read()
                    documents.append(f"File: {filepath}\n{content[:2000]}")
            except:
                pass

        documents.append("""
        This project implements a skin lesion analysis system based on the
        1st and 2nd place solution from ISIC 2024 competition.
        It includes an ensemble of vision models, GBDT blending,
        and a FastAPI backend.
        """)

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=50)
        texts = text_splitter.create_documents(documents)

        self.vectorstore = FAISS.from_documents(texts, self.embeddings)
        print(f"Indexed {len(texts)} document chunks")
        return self.vectorstore

    def ask(self, question: str):
        if self.vectorstore is None:
            self.index_project()

        docs = self.vectorstore.similarity_search(question, k=3)
        context = "\n\n".join([doc.page_content[:400] for doc in docs])

        return f"""Context from project:\n{context[:600]}...\n\nQuestion: {question}"""


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/api/rag_routes.py
"""
Routes of RAG Assistant
"""

from fastapi import APIRouter
from pydantic import BaseModel
from src.rag.rag_engine import RAGEngine

rag_router = APIRouter()
rag_engine = RAGEngine()

class ChatRequest(BaseModel):
    question: str

class ChatResponse(BaseModel):
    answer: str

@rag_router.post("/chat", response_model=ChatResponse)
async def chat_with_assistant(request: ChatRequest):
    """Chat with the RAG assistant"""
    answer = rag_engine.ask(request.question)
    return ChatResponse(answer=answer)



In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/api/main.py
"""
Main FastAPI application
"""

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

from src.core.config import get_settings
from src.api.routes import health_router, prediction_router
from src.api.rag_routes import rag_router

settings = get_settings()

app = FastAPI(
    title=settings.APP_NAME,
    version=settings.API_VERSION,
    description="ISIC 2024 Skin Cancer Detection API"
)

app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

app.include_router(health_router, prefix="/api/v1", tags=["health"])
app.include_router(prediction_router, prefix="/api/v1", tags=["prediction"])
app.include_router(rag_router, prefix="/api/v1", tags=["rag"])   # ← Added

@app.get("/")
async def root():
    return {"message": "ISIC 2024 Flagship API is running"}


In [ ]:
!pip install -U nbformat nbconvert jupyter

In [ ]:
!python - <<'PY'
import nbformat

path = "Rag_assistant1.ipynb"

try:
    with open(path, "r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)

    nbformat.validate(nb)
    print("Notebook is valid.")
except Exception as e:
    print("Notebook has a problem:")
    print(type(e).__name__)
    print(e)


In [ ]:
import nbformat

path = "Rag_assistant1.ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

for cell in nb.cells:
    if cell.cell_type == "code":
        cell.outputs = []
        cell.execution_count = None

nb.metadata.pop("widgets", None)

with open(path, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Cleaned notebook outputs and widget metadata.")

In [ ]:
!jupyter nbconvert --to html Rag_assistant1.ipynb

In [ ]:
pwd
ls *.ipynb